# {slug} — notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e consistente col precedente
- non sostituisce l'analisi pubblica (quella va in `analisi-civiche`)
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import pandas as pd
from pathlib import Path

SLUG = "{slug}"
ANNO_RUN = {anno_run}           # anno di riferimento del run
MART_TABLE = "{mart_table_name}"
METRICA = "{metrica_guida}"    # colonna numerica principale del mart

# Risoluzione candidate dir (funziona sia da notebooks/ che dalla root candidate)
candidate_dir = Path.cwd().resolve()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError(f"dataset.yml non trovato risalendo da {Path.cwd()}")

DI_ROOT = candidate_dir.parents[1]
RAW_DIR   = DI_ROOT / "out" / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "out" / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "out" / "data" / "mart"  / SLUG / str(ANNO_RUN)

print(f"candidate : {candidate_dir.name}")
print(f"raw       : {RAW_DIR}")
print(f"clean     : {CLEAN_DIR}")
print(f"mart      : {MART_DIR}")

## 1. Raw

Verifica che il file raw esista e abbia la forma attesa dalla fonte.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File raw: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

if raw_path.suffix == ".csv":
    raw = pd.read_csv(raw_path, nrows=5)
elif raw_path.suffix == ".xlsx":
    raw = pd.read_excel(raw_path, nrows=5)
else:
    raw = pd.read_parquet(raw_path).head(5)

print(f"Colonne raw: {list(raw.columns)}")
raw

## 2. Clean

Verifica schema, null e range sul layer clean.

In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire prima: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
print(f"Shape clean: {clean.shape}")
print(f"Colonne    : {list(clean.columns)}")
clean.head()

In [ ]:
print("Dtypes:")
print(clean.dtypes.to_string())
print("\nNull per colonna:")
print(clean.isnull().sum().to_string())

## 3. Mart

Verifica aggregazioni e consistenza con il layer clean.

In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire prima: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Shape mart: {mart.shape}")
print(f"Colonne   : {list(mart.columns)}")
mart.head()

In [ ]:
print("Null per colonna:")
print(mart.isnull().sum().to_string())

if METRICA in mart.columns and pd.api.types.is_numeric_dtype(mart[METRICA]):
    negativi = int((mart[METRICA] < 0).sum())
    print(f"\nRange {METRICA}: min={mart[METRICA].min():.2f}, max={mart[METRICA].max():.2f}, negativi={negativi}")

In [ ]:
# Consistenza mart vs clean: il totale della metrica deve coincidere
# Adattare METRICA_CLEAN al nome della colonna corrispondente nel clean
METRICA_CLEAN = "{metrica_clean}"  # es. "previsioni_definitive_cp"

if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_mart - tot_clean) / tot_clean * 100 if tot_clean != 0 else float('nan')
    print(f"Totale mart  ({METRICA})      : {tot_mart:.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    assert delta_pct < 0.01, f"Delta troppo alto: {delta_pct:.2f}% — verificare la pipeline"
    print("OK: i totali coincidono.")
else:
    print(f"Colonne non trovate per il check di consistenza — aggiornare METRICA e METRICA_CLEAN.")

## Note v0

- Slug: `{slug}`
- Anno run: `{anno_run}`
- Tabella mart: `{mart_table_name}`
- Metrica guida: `{metrica_guida}`
- Perimetro: {perimetro_v0}